---
title: Options
description: Specify options when building with the Sampler primitive.
---


# Sampler options

{/*
  DO NOT EDIT THIS CELL!!!
  This cell's content is generated automatically by a script. Anything you add
  here will be removed next time the notebook is run. To add new content, create
  a new cell before or after this one.
*/}

<details>
<summary><b>Package versions</b></summary>

The code on this page was developed using the following requirements.
We recommend using these versions or newer.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>

You can use options to customize the Sampler primitive. This section focuses on how to specify Qiskit Runtime primitive options. While the interface of the primitives' `run()`  method is common across all implementations, their options are not. Consult the corresponding API references for information about the [`qiskit.primitives.BackendSamplerV2`](/docs/api/qiskit/qiskit.primitives.BackendSamplerV2) and [`qiskit_aer.primitives.SamplerV2`](https://qiskit.github.io/qiskit-aer/stubs/qiskit_aer.primitives.SamplerV2.html) options.

<Admonition type="note" title="Notes about specifying options in the Sampler primitives">
- You can see the available options and update option values during or after primitive initialization.
- Use the `update()` method to apply changes to the `options` attribute.
- If you do not specify a value for an option, it is given a special value of `Unset` and the server defaults are used.
- The `options` attribute is the `dataclass` Python type.  You can use the built-in `asdict` method to convert it to a dictionary.
</Admonition>

<span id="pass-options"></span>
## Set Sampler options

You can set options when initializing the primitive, after initializing the primitive, or (for `shots` only), in the `run()` method.

### Primitive initialization

You can pass in an instance of the options class or a dictionary when initializing Sampler, which then makes a copy of those options. Thus, changing the original dictionary or options instance doesn't affect the options owned by the primitive.

#### Options class

When creating an instance of the `SamplerV2` class, you can pass in an instance of the options class. Those options will then be applied when you use `run()` to perform the calculation.  Specify the options in this format:  `options.option.sub-option.sub-sub-option = choice`.  For example: `options.dynamical_decoupling.enable = True`

See [`SamplerOptions`](/docs/api/qiskit-ibm-runtime/options-sampler-options) for full details about the options class.

In [4]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit_ibm_runtime.options import SamplerOptions

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

options = SamplerOptions(
    dynamical_decoupling={"enable": True, "sequence_type": "XpXm"},
)

# or...
options = SamplerOptions()
options.dynamical_decoupling.enable = True
options.dynamical_decoupling.sequence_type = "XpXm"

sampler = Sampler(mode=backend, options=options)

#### Dictionary

You can specify options as a dictionary when initializing Sampler.

In [6]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2 as Sampler

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Setting options during primitive initialization
sampler = Sampler(
    backend,
    options={
        "dynamical_decoupling": {
            "enable": True,
            "sequence_type": "XpXm",
        },
    },
)

### Update options after initialization

You can specify the options in this format: `sampler.options.option.sub-option.sub-sub-option = choice` to take advantage of auto-complete, or use the `update()` method to make bulk updates.

The `SamplerV2` options class ([`SamplerOptions`](/docs/api/qiskit-ibm-runtime/options-sampler-options)) does not need to be instantiated if you are setting options after initializing the primitive.

In [7]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2 as Sampler

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

sampler = Sampler(mode=backend)

# Setting options after primitive initialization
# This uses auto-complete.
sampler.options.default_shots = 4000
# This does bulk update.
sampler.options.update(
    default_shots=4000,
    dynamical_decoupling={"enable": True, "sequence_type": "XpXm"},
)

<span id="run-method"></span>
### Run() method

The only values you can pass to `run()` are those defined in the interface.  That is, `shots`. This overwrites any value set for `default_shots` for the current run.

In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit.circuit.library import random_iqp
from qiskit.transpiler import generate_preset_pass_manager

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

circuit1 = random_iqp(3)
circuit1.measure_all()
circuit2 = random_iqp(3)
circuit2.measure_all()

pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend
)

transpiled1 = pass_manager.run(circuit1)
transpiled2 = pass_manager.run(circuit2)

sampler = Sampler(mode=backend)
# Default shots to use if not specified in run()
sampler.options.default_shots = 500
# Sample two circuits at 128 shots each.
sampler.run([transpiled1, transpiled2], shots=128)

<RuntimeJobV2('d7ans0b0g7hs73doium0', 'sampler')>

### Special cases

<span id="shots"></span>
#### Shots

The `SamplerV2.run` method accepts two arguments: a list of PUBs, each of which can specify a PUB-specific value for shots, and a shots keyword argument. These shot values are a part of the Sampler execution interface, and are independent of the Runtime Sampler's options.  They take precedence over any values specified as options in order to comply with the Sampler abstraction.

However, if `shots` is not specified by any PUB or in the run keyword argument (or if they are all `None`), then the shots value from the options is used, most notably `default_shots`.

To summarize, this is the order of precedence for specifying shots in the Sampler, for any particular PUB:

1. If the PUB specifies shots, use that value.
2. If the `shots` keyword argument is specified in `run`, use that value.
3. If `num_randomizations` and `shots_per_randomization` are specified as `twirling` options, shots are the product of those values.
3. If `sampler.options.default_shots` is specified, use that value.

Thus, if shots are specified in all possible places, the one with highest precedence (shots specified in the PUB) is used.

Example:

In [6]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit.circuit.library import random_iqp
from qiskit.transpiler import generate_preset_pass_manager

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

circuit1 = random_iqp(3)
circuit1.measure_all()
circuit2 = random_iqp(3)
circuit2.measure_all()

pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend
)

transpiled1 = pass_manager.run(circuit1)
transpiled2 = pass_manager.run(circuit2)


# Setting shots during primitive initialization
sampler = Sampler(mode=backend, options={"default_shots": 4096})

# Setting options after primitive initialization
# This uses auto-complete.
sampler.options.default_shots = 2000

# This does bulk update.  The value for default_shots is overridden if you specify shots with run() or in the PUB.
sampler.options.update(
    default_shots=1024, dynamical_decoupling={"sequence_type": "XpXm"}
)

# Sample two circuits at 128 shots each.
sampler.run([transpiled1, transpiled2], shots=128)

<RuntimeJobV2('d5k96icjt3vs73ds5t0g', 'sampler')>

<span id="options-table"></span>
## Available options

The following table documents options from the latest version of `qiskit-ibm-runtime`. To see older option versions, visit the [`qiskit-ibm-runtime` API reference](/docs/api/qiskit-ibm-runtime) and select a previous version.


<Accordion>
<AccordionItem title="`default_shots`">

The total number of shots to use per circuit per configuration.

**Choices**: Integer >= 0

**Default**: None

[`default_shots` API documentation](/docs/api/qiskit-ibm-runtime/options-sampler-options#default_shots)
  </AccordionItem>


<AccordionItem title="`dynamical_decoupling`">

Control dynamical decoupling error mitigation settings.

[`dynamical_decoupling` API documentation](/docs/api/qiskit-ibm-runtime/options-sampler-options#dynamical_decoupling)
<Accordion>
<AccordionItem title="`dynamical_decoupling.enable`">

**Choices**: `True`, `False`

**Default**: `False`
  </AccordionItem>


<AccordionItem title="`dynamical_decoupling.extra_slack_distribution`">

**Choices**: `middle`, `edges`

**Default**: `middle`
  </AccordionItem>


<AccordionItem title="`dynamical_decoupling.scheduling_method`">

Choices: `asap`, `alap`
Default: `alap`
  </AccordionItem>


<AccordionItem title="`dynamical_decoupling.sequence_type`">

Choices: `XX`, `XpXm`, `XY4`
Default: `XX`
  </AccordionItem>


<AccordionItem title="`dynamical_decoupling.skip_reset_qubits`">

Choices: `True`, `False`
Default: `False`
  </AccordionItem>

</Accordion>

  </AccordionItem>



<AccordionItem title="`environment`">
[`environment` API documentation](/docs/api/qiskit-ibm-runtime/options-sampler-options#environment)
<Accordion>
<AccordionItem title="`environment.job_tags`">

List of tags.

**Choices**: None

**Default**: None
  </AccordionItem>


<AccordionItem title="`environment.log_level`">

**Choices**: DEBUG, INFO, WARNING, ERROR, CRITICAL

**Default**: WARNING
  </AccordionItem>


<AccordionItem title="`environment.private`">

**Choices**: `True`, `False`

**Default**: `False`
  </AccordionItem>

</Accordion>

  </AccordionItem>



<AccordionItem title="`execution`">
[`execution` API documentation](/docs/api/qiskit-ibm-runtime/options-sampler-options#execution)
<Accordion>
<AccordionItem title="`execution.init_qubits`">
Whether to reset the qubits to the ground state for each shot.

**Choices**: `True`, `False`

**Default**: `True`
  </AccordionItem>


<AccordionItem title="`execution.rep_delay`">
The delay between a measurement and the subsequent quantum circuit.

**Choices**: Value in the range supplied by `backend.rep_delay_range`

**Default**: Given by `backend.default_rep_delay`
  </AccordionItem>


<AccordionItem title="`execution.meas_type`">

**Choices**: `classified`, `kerneled`, `avg_kerneled`

**Default**: `classified`
  </AccordionItem>

</Accordion>

  </AccordionItem>



<AccordionItem title="`max_execution_time`">
Limits how long a job can run, in seconds. See the [maximum execution time](/docs/guides/max-execution-time) guide for details.

**Choices**: Integer number of seconds in the range [1, 10800] 

**Default**: 10800 (3 hours)

[`max_execution_time` API documentation](/docs/api/qiskit-ibm-runtime/options-sampler-options#max_execution_time)
  </AccordionItem>


<AccordionItem title="`simulator`">

Options to pass when simulating a backend

[`simulator` API documentation](/docs/api/qiskit-ibm-runtime/options-simulator-options)
<Accordion>
<AccordionItem title="`simulator.basis_gates`">

**Choices**: List of basis gate names to unroll to

**Default**: The set of all basis gates supported by [Qiskit Aer simulator](https://qiskit.github.io/qiskit-aer/stubs/qiskit_aer.AerSimulator.html)

  </AccordionItem>



<AccordionItem title="`simulator.coupling_map`">

**Choices**: List of directed two-qubit interactions

**Default**: None, which implies no connectivity constraints (full connectivity).

  </AccordionItem>



<AccordionItem title="`simulator.noise_model`">

**Choices**: [Qiskit Aer NoiseModel](/docs/guides/build-noise-models), or its representation

**Default**: None

  </AccordionItem>



<AccordionItem title="`simulator.seed_simulator`">

**Choices**: Integer

**Default**: None

  </AccordionItem>


</Accordion>

  </AccordionItem>



<AccordionItem title="`twirling`">

Twirling options

[`twirling` API documentation](/docs/api/qiskit-ibm-runtime/options-twirling-options)
<Accordion>
<AccordionItem title="`twirling.enable_gates`">

**Choices**: True, False

**Default**: False

  </AccordionItem>



<AccordionItem title="`twirling.enable_measure`">

**Choices**: True, False

**Default**: False

  </AccordionItem>



<AccordionItem title="`twirling.num_randomizations`">

**Choices**: `auto`, Integer >= 1

**Default**: `auto`

  </AccordionItem>



<AccordionItem title="`twirling.shots_per_randomization`">

**Choices**: `auto`, Integer >= 1

**Default**: `auto`

  </AccordionItem>



<AccordionItem title="`twirling.strategy`">

**Choices**: `active`, `active-circuit`, `active-accum`, `all`

**Default**: `active-accum`

  </AccordionItem>


</Accordion>

  </AccordionItem>



<AccordionItem title="`experimental`">

Experimental options, when available.

  </AccordionItem>


</Accordion>

<span id="options-compatibility-table"></span>
## Feature compatibility

Due to differences in the device compilation process, certain runtime features cannot be used together in a single job. Click the appropriate tab for a list of features that are incompatible with the selected feature:

<Tabs>
  <TabItem value="Dynamic" label="Dynamic circuits">
  Incompatible with:
  - Dynamical decoupling

  Other notes:
  - Gate twirling can be applied to dynamic circuits, but only to gates not inside conditional blocks. Measurement twirling can only be applied to terminal measurements.
  - Compatible with fractional gates when using `qiskit-ibm-runtime` v0.42.0 or later.

  </TabItem>
  <TabItem value="DD" label="Dynamical decoupling">
  Incompatible with dynamic circuits.

  </TabItem>

  <TabItem value="Fractional" label="Fractional gates">
  Incompatible with:
  - Gate twirling

  Compatible with dynamic circuits when using `qiskit-ibm-runtime` v0.42.0 or later.

  </TabItem>

  <TabItem value="Twirling" label="Gate twirling">
  Incompatible with fractional gates or with stretches.

  Other notes:
  - Gate twirling can be applied to dynamic circuits, but only to gates not inside conditional blocks. Measurement twirling can only be applied to terminal measurements.
  - Does not work with non-Clifford entanglers.

  </TabItem>


</Tabs>

## Next steps

<Admonition type="tip" title="Recommendations">
    - Find more details about the `SamplerV2` methods in the [Sampler API reference](../api/qiskit-ibm-runtime/sampler-v2).
    - Find details about how to configure [error suppression](configure-error-suppression) and [error mitigation](configure-error-mitigation).
    - Decide what [execution mode](execution-modes) to run your job in.
</Admonition>